# NOOTEBOOK DE CORTE/CLIP A ZONA AGRICOLA

## EN ESTE NOOTEBOOK REALIZAREMOS EL CORTE DE TODOS LOS RASTER POR EL POLIGONO AGRICOLA DE LA LOCALIDAD DE MARCOS JUAREZ.
ASD}

In [ ]:
#!/usr/bin/env python3
"""
Clip/Mask de mosaics usando shapefile de zonas agrícolas.
Por: Usuario -> Marcos Juarez workflow

Comportamiento por defecto:
 - Para cada TIFF en MOSAIC_DIR:
    1) selecciona sólo las geometrías del shapefile que intersectan el raster
    2) realiza rasterio.mask.mask(keep_inside=True)
    3) guarda el resultado recortado al bounding box de la(s) geometría(s) (CROP=True)
    4) output filename: <origname>_clipped.tif
"""

In [1]:


import os
import glob
from tqdm import tqdm
import rasterio
from rasterio.mask import mask
import geopandas as gpd
from shapely.geometry import mapping, box
import numpy as np



# ---------- CONFIG ----------
MOSAIC_DIR = r"E:\Silos\Base de datos\procesado_sin_nubes\mosaics"   # carpeta con los 328 mosaics
SHAPEFILE = r"C:\Users\m\Documents\Tesis_Silos\Silos_FR_MCDtf\GeoDatos\argentina_Agriculture_MERGE_Marcos_Juarez.shp"
OUT_DIR = r"E:\Silos\Base de datos\procesado_sin_nubes\mosaics\mosaics_clipped"  # salida
CROP = True            # True -> recorta al bbox de la(s) geometría(s); False -> conserva tamaño original (mask con nodata fuera)
NODATA = np.nan        # valor nodata de salida (puedes cambiarlo)
COMPRESS = "LZW"       # compresión para GeoTIFF de salida (None para sin compresión)
OVERWRITE = False      # permitir sobreescribir archivos ya existentes
# ----------------------------

os.makedirs(OUT_DIR, exist_ok=True)

# leer shapefile (geodataframe)
gdf = gpd.read_file(SHAPEFILE)
if gdf.empty:
    raise SystemExit(f"Shapefile {SHAPEFILE} cargado pero vacío.")

# intentar reparar geometrías simples si necesario
gdf['geometry'] = gdf['geometry'].buffer(0)

# lista de mosaics
patterns = [os.path.join(MOSAIC_DIR, "*.tif"), os.path.join(MOSAIC_DIR, "*.tiff")]
rasters = []
for p in patterns:
    rasters.extend(glob.glob(p))
rasters = sorted(rasters)
print(f"Se encontraron {len(rasters)} rasters en {MOSAIC_DIR}.")

def choose_fill_for_mask(src_dtype_str, desired_nodata):
    """
    Elige un fill_value compatible con src_dtype para pasar a rasterio.mask.mask.
    Si desired_nodata cabe en src_dtype, lo usa. Si no:
      - para unsigned int y desired_nodata < 0 -> usa max del dtype (e.g. 65535)
      - para otros casos intenta usar un valor compatible (min-1 o max) conservador
    Devuelve (fill_value, out_dtype_after_postprocess, use_nan_flag)
    - out_dtype_after_postprocess: dtype string que usaremos al guardar el resultado
    - use_nan_flag: True si queremos convertir a float y usar np.nan para nodata
    """
    np_dtype = np.dtype(src_dtype_str)
    use_nan = False

    # caso float: dejamos usar np.nan si desired_nodata is np.nan
    if np.issubdtype(np_dtype, np.floating):
        if desired_nodata is np.nan:
            return (np.nan, 'float32', True)
        else:
            # si pediste -9999 y el raster es float, -9999 es válido
            return (desired_nodata, 'float32', False)

    # entero
    if np.issubdtype(np_dtype, np.integer):
        info = np.iinfo(np_dtype)
        # si desired_nodata es NaN -> vamos a usar float output
        if desired_nodata is np.nan:
            # elegimos un fill compatible (por ejemplo max) y luego convertiremos a np.nan en float32
            fill = info.max
            return (int(fill), 'float32', True)

        # desired_nodata es número (entero negativo o positivo)
        try:
            d = int(desired_nodata)
        except Exception:
            # fallback
            fill = info.max
            return (int(fill), 'float32', True)

        # si cabe en dtype
        if info.min <= d <= info.max:
            # se puede usar directamente (mantener dtype de salida similar al original para tamaño)
            # elegimos out dtype: si dtype es unsigned but desired_nodata < 0 -> no cabe (fallaría)
            if np.issubdtype(np_dtype, np.unsignedinteger) and d < 0:
                # no cabe -> usaremos out dtype int32 y usaremos fill=temp_max, luego reemplazamos por desired_nodata
                fill = info.max
                return (int(fill), 'int32', False)
            else:
                return (d, src_dtype_str, False)
        else:
            # desired_nodata no cabe en este dtype
            if np.issubdtype(np_dtype, np.unsignedinteger) and d < 0:
                fill = info.max
                return (int(fill), 'int32', False)
            else:
                # caso raro: pediste 100000 pero dtype uint16 -> usar max como fill y luego cast a int32 y reemplazar
                fill = info.max
                return (int(fill), 'int32', False)

    # fallback general
    return (0, 'float32', True)

def geometries_intersecting_raster(raster_path, gdf_shp):
    """
    Devuelve lista de geometrías (shapely) del gdf_shp que intersectan el raster bounds.
    Reproyecta geometrías al CRS del raster si es necesario.
    Usa spatial index (sindex) si está disponible, con fallback a filtro por intersección.
    """
    with rasterio.open(raster_path) as src:
        rast_crs = src.crs
        bounds = src.bounds  # left, bottom, right, top
        src_dtype = src.dtypes[0]

    # reproyectar si hace falta
    if gdf_shp.crs != rast_crs:
        try:
            gdf_rast = gdf_shp.to_crs(rast_crs)
        except Exception as e:
            print(f"[WARN] Falló reproyección del shapefile a CRS del raster ({e}). Se saltea raster.")
            return [], src_dtype
    else:
        gdf_rast = gdf_shp

    # crear bbox como shapely.geometry.box
    bbox_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

    # intentar usar spatial index para filtrar rápido
    try:
        sindex = gdf_rast.sindex
        if sindex is not None:
            possible_idx = list(sindex.intersection(bbox_geom.bounds))
            if len(possible_idx) == 0:
                return [], src_dtype
            possible = gdf_rast.iloc[possible_idx]
        else:
            possible = gdf_rast
    except Exception:
        possible = gdf_rast

    # ahora filtrar por intersección real
    try:
        intersects = possible[possible.geometry.intersects(bbox_geom)]
    except Exception:
        intersects = possible[possible.geometry.apply(lambda g: g.intersects(bbox_geom))]

    if intersects.empty:
        return [], src_dtype

    return list(intersects.geometry.values), src_dtype

# Procesamiento principal
for rast_path in tqdm(rasters, desc="Procesando rasters"):
    fname = os.path.splitext(os.path.basename(rast_path))[0]
    out_path = os.path.join(OUT_DIR, f"{fname}_clipped.tif")
    if os.path.exists(out_path) and not OVERWRITE:
        continue

    geoms, src_dtype = geometries_intersecting_raster(rast_path, gdf)
    if len(geoms) == 0:
        print(f"[SKIP] Ningún polígono intersecta {os.path.basename(rast_path)}. Se saltea.")
        continue

    shapes = [mapping(g) for g in geoms]

    with rasterio.open(rast_path) as src:
        meta = src.meta.copy()

        # determinamos fill_value compatible con dtype origen y la estrategia de salida
        fill_value, out_dtype_after_post, use_nan_flag = choose_fill_for_mask(src_dtype, NODATA)

        # Ejecutamos mask con fill_value que sí cabe en el dtype de origen
        try:
            out_image, out_transform = mask(dataset=src, shapes=shapes, invert=False, crop=CROP, nodata=fill_value)
        except Exception as e:
            print(f"[ERROR] Al aplicar mask en {rast_path}: {e}")
            continue

        # Ahora convertimos post-process:
        # - si queremos np.nan como nodata: pasar a float32 y reemplazar fill_value -> np.nan
        # - si queremos un NODATA entero (ej -9999) pero origen era uint16: casteamos a int32 y reemplazamos fill_value -> NODATA
        if use_nan_flag:
            # convert to float32 and set fill_value -> np.nan
            out_image = out_image.astype(np.float32)
            # comparaciones directas: fill_value puede ser large int; reemplazamos igual
            mask_fill = (out_image == float(fill_value))
            # si el raster tenía valores legítimos iguales a fill_value, esto los convertirá a nan también;
            # normalmente fill_value es un valor extremo improbable (ej 65535) así que está bien.
            out_image[mask_fill] = np.nan
            final_dtype = 'float32'
            final_nodata = np.nan
        else:
            # desired_nodata is int-like (e.g., -9999 or fits in dtype)
            if isinstance(NODATA, float) and np.isnan(NODATA):
                # user asked np.nan but choose_fill returned use_nan False (rare) -> force float conversion
                out_image = out_image.astype(np.float32)
                out_image[out_image == float(fill_value)] = np.nan
                final_dtype = 'float32'
                final_nodata = np.nan
            else:
                # queremos nodo entero: si out_dtype_after_post is not same as current dtype, casteamos
                if out_image.dtype != np.dtype(out_dtype_after_post):
                    # casteo
                    out_image = out_image.astype(np.dtype(out_dtype_after_post))
                # reemplazamos fill_value por NODATA (ej -9999)
                out_image[out_image == fill_value] = NODATA
                final_dtype = out_dtype_after_post
                final_nodata = NODATA

        # preparar metadata de salida
        out_meta = meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "dtype": final_dtype,
            "count": out_image.shape[0],
            "compress": COMPRESS,
            "nodata": final_nodata
        })

        # escribir archivo (si final_dtype es 'float32' y nodata np.nan, rasterio lo escribirá como float)
        try:
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)
        except Exception as e:
            print(f"[ERROR] Al guardar {out_path}: {e}")
            continue

print("Proceso completado.")


Se encontraron 328 rasters en E:\Silos\Base de datos\procesado_sin_nubes\mosaics.


Procesando rasters: 100%|██████████| 328/328 [13:42:37<00:00, 150.48s/it]  

Proceso completado.
